In [1]:
import pandas as pd
import re
import csv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
def limpar_texto(text):
    if not isinstance(text, str):
        return ""
    # Remover tags HTML que aparecem muito em phishings
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remover URLs explícitas para o modelo focar na linguagem do texto
    text = re.sub(r'http\s+|www\.\S+', ' [url] ', text)
    # Remover quebras de linha, tabulações e múltiplos espaços
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# ==========================================
# 1. CARREGAR E PADRONIZAR NAZARIO
# ==========================================
print("Carregando Nazario...")
try:
    df_naz = pd.read_csv("Nazario_5.csv", engine='python', on_bad_lines='skip')

    # Identificar coluna de texto do Nazario dinamicamente
    col_texto_naz = None
    for col in df_naz.columns:
        if col.lower() in ['body', 'text', 'message', 'email', 'email text']:
            col_texto_naz = col
            break
    if not col_texto_naz:
        col_texto_naz = df_naz.select_dtypes(include=['object']).columns[0]

    df_naz_clean = pd.DataFrame({
        'text': df_naz[col_texto_naz].apply(limpar_texto),
        'label': 1  # Nazario é majoritariamente/totalmente phishing
    })
    print(f"Nazario carregado com sucesso! Formato: {df_naz_clean.shape}")
except Exception as e:
    print(f"Erro ao carregar Nazario: {e}")

# ==========================================
# 2. CARREGAR E PADRONIZAR ENRON (SOLUÇÃO ROBUSTA)
# ==========================================
print("\nCarregando Enron...")
try:
    # Usando o motor em Python e pulando linhas malformadas que causavam o EOF error
    df_enron = pd.read_csv("enron_fraud.csv", engine='python', on_bad_lines='skip')
    print("Enron lido pelo interpretador Python.")
except Exception as e:
    print(f"Aviso no motor Python: {e}. Tentando fallback com codificação alternativa...")
    df_enron = pd.read_csv("enron_fraud.csv", encoding='utf-8', encoding_errors='replace', on_bad_lines='skip')

# Identificação dinâmica das colunas do Enron para evitar KeyError
col_texto_enron = None
for col in df_enron.columns:
    if col.lower() in ['body', 'text', 'message', 'email', 'content']:
        col_texto_enron = col
        break
if not col_texto_enron:
    col_texto_enron = df_enron.select_dtypes(include=['object']).columns[0]

col_label_enron = None
for col in df_enron.columns:
    if col.lower() in ['target', 'label', 'fraud', 'class']:
        col_label_enron = col
        break
if not col_label_enron:
    col_label_enron = df_enron.select_dtypes(include=['number']).columns[0]

print(f"Colunas identificadas no Enron -> Texto: '{col_texto_enron}' | Label: '{col_label_enron}'")

df_enron_clean = pd.DataFrame({
    'text': df_enron[col_texto_enron].apply(limpar_texto),
    'label': df_enron[col_label_enron].astype(int)
})
print(f"Enron processado com sucesso! Formato: {df_enron_clean.shape}")

# ==========================================
# 3. FUSÃO E TRATAMENTO DE DESBALANCEAMENTO
# ==========================================
print("\nUnificando os conjuntos de dados...")
df_completo = pd.concat([df_naz_clean, df_enron_clean], ignore_index=True)

# Remover linhas vazias ou nulas geradas pós-limpeza
df_completo.dropna(subset=['text'], inplace=True)
df_completo = df_completo[df_completo['text'] != ""]

print(f"Distribuição total das classes combinadas:\n{df_completo['label'].value_counts()}")

# Amostragem equilibrada (Undersampling da classe majoritária)
min_samples = df_completo['label'].value_counts().min()
df_balanced = df_completo.groupby('label').sample(n=min_samples, random_state=42).reset_index(drop=True)

print(f"\nDistribuição final após o balanceamento:\n{df_balanced['label'].value_counts()}")

Carregando Nazario...
Nazario carregado com sucesso! Formato: (3063, 2)

Carregando Enron...
Enron lido pelo interpretador Python.
Colunas identificadas no Enron -> Texto: 'Body' | Label: 'Label'
Enron processado com sucesso! Formato: (447274, 2)

Unificando os conjuntos de dados...
Distribuição total das classes combinadas:
0    443574
1      5388
Name: label, dtype: int64

Distribuição final após o balanceamento:
0    5388
1    5388
Name: label, dtype: int64


In [4]:
# Mostra a lista exata com o nome de todas as colunas presentes
print("Colunas presentes no dataset unificado:")
print(df_completo.columns.tolist())

# Mostra o formato do dataset (Número de linhas, Número de colunas)
print(f"\nFormato do dataset unificado: {df_completo.shape}")

Colunas presentes no dataset unificado:
['text', 'label']

Formato do dataset unificado: (448962, 2)


In [5]:
X = df_balanced["text"]
y = df_balanced["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Treino:", len(X_train))
print("Teste :", len(X_test))

Treino: 8620
Teste : 2156


In [6]:
!pip install transformers datasets torch

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

/home/gaby/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
email = """
URGENTE: Sua conta será suspensa.
Clique aqui para verificar seus dados.
"""

inputs = tokenizer(
    email,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

In [10]:
!pip install transformers accelerate torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "meta-llama/Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

OSError: meta-llama/Llama-3-8B-Instruct is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
prompt = f"""
Analise o email abaixo.

Retorne:
- phishing ou legítimo
- justificativa

Email:

URGENTE! Atualize sua senha agora.
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

In [12]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="microsoft/deberta-v3-base"
)

result = classifier(
    "URGENTE: clique aqui para redefinir sua senha"
)

print(result)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ValueError: Converting from Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast convertors: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Tokenizer', 'RealmTokenizer', 'Refor